In [1]:
import cv2 as cv
import numpy as np
import os
from datetime import datetime
import geocoder

try:
    x = datetime.now().strftime('%Y-%m-%d %H-%M-%S')
    print(x)

    class_name = []
    with open(r'utils/obj.names', 'r') as f:
        class_name = [cname.strip() for cname in f.readlines()]

    net1 = cv.dnn.readNet(r'utils/yolov4_tiny.weights', r'utils/yolov4_tiny.cfg')
    net1.setPreferableBackend(cv.dnn.DNN_BACKEND_CUDA)
    net1.setPreferableTarget(cv.dnn.DNN_TARGET_CUDA_FP16)
    model1 = cv.dnn_DetectionModel(net1)
    model1.setInputParams(size=(640, 480), scale=1/255, swapRB=True)

    image_path = "test/Pothole/1.jpg"  # Change this to the input image path
    frame = cv.imread(image_path)
    if frame is None:
        raise Exception("Failed to load image")

    height, width, _ = frame.shape
    result_path = "pothole_coordinates"
    Conf_threshold = 0.5
    NMS_threshold = 0.4

    # Define the region of interest (ROI) mask
    mask = np.zeros_like(frame)
    mask[0:int(0.85 * height), :] = 255

    # Apply the mask to the frame
    masked_frame = cv.bitwise_and(frame, mask)

    classes, scores, boxes = model1.detect(masked_frame, Conf_threshold, NMS_threshold)
    g = geocoder.ip('me')
    for (classid, score, box) in zip(classes, scores, boxes):
        label = "pothole"
        x, y, w, h = box
        recarea = w * h
        area = width * height

        severity = ""
        severity_threshold_low = 0.007  # Adjust as needed
        severity_threshold_medium = 0.020  # Adjust as needed

        if len(scores) != 0 and scores[0] >= 0.7:
            if (recarea / area) <= severity_threshold_low:
                severity = "Low"
            elif (recarea / area) <= severity_threshold_medium:
                severity = "Medium"
            else:
                severity = "High"

            if severity != "":
                cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 1)
                cv.putText(frame, f"%{round(scores[0] * 100, 2)} {label} ({severity} Severity)",
                           (box[0], box[1] - 10), cv.FONT_HERSHEY_COMPLEX, 0.5, (255, 0, 0), 1)

                output_image_path = os.path.join(result_path, 'detected_pothole.jpg')
                cv.imwrite(output_image_path, frame)
                with open(os.path.join(result_path, 'detected_pothole.txt'), 'w') as f:
                    f.write(f"{str(g.latlng)}\nSeverity: {severity}")

    cv.imshow('Pothole Detection', frame)
    cv.waitKey(0)
    cv.destroyAllWindows()

except Exception as e:
    print(f"Error: {e}")


2026-07-11 00-39-23
Error: Failed to load image


In [2]:
import cv2 as cv
import numpy as np
import os
from datetime import datetime
import geocoder

try:
    x = datetime.now().strftime('%Y-%m-%d %H-%M-%S')
    print(x)

    class_name = []
    with open(r'utils/obj.names', 'r') as f:
        class_name = [cname.strip() for cname in f.readlines()]

    net1 = cv.dnn.readNet(r'utils/yolov4_tiny.weights', r'utils/yolov4_tiny.cfg')
    net1.setPreferableBackend(cv.dnn.DNN_BACKEND_CUDA)
    net1.setPreferableTarget(cv.dnn.DNN_TARGET_CUDA_FP16)
    model1 = cv.dnn_DetectionModel(net1)
    model1.setInputParams(size=(640, 480), scale=1/255, swapRB=True)

    image_path = "test/Pothole/1.jpg"  # Change this to the input image path
    frame = cv.imread(image_path)
    if frame is None:
        raise Exception("Failed to load image")

    height, width, _ = frame.shape
    result_path = "pothole_coordinates"
    os.makedirs(result_path, exist_ok=True)  # Ensure the directory exists

    Conf_threshold = 0.5
    NMS_threshold = 0.4

    # Define the region of interest (ROI) mask
    mask = np.zeros_like(frame)
    mask[0:int(0.85 * height), :] = 255

    # Apply the mask to the frame
    masked_frame = cv.bitwise_and(frame, mask)

    classes, scores, boxes = model1.detect(masked_frame, Conf_threshold, NMS_threshold)
    g = geocoder.ip('me')

    total_pothole_area = 0  # Store total detected pothole area
    pothole_data = []  # Store detected pothole details

    for (classid, score, box) in zip(classes, scores, boxes):
        label = "pothole"
        x, y, w, h = box
        recarea = w * h  # Area of the detected pothole
        total_pothole_area += recarea  # Sum up the pothole areas
        image_area = width * height

        severity = ""
        severity_threshold_low = 0.007  # Adjust as needed
        severity_threshold_medium = 0.020  # Adjust as needed

        if score >= 0.7:
            relative_area = recarea / image_area
            if relative_area <= severity_threshold_low:
                severity = "Low"
            elif relative_area <= severity_threshold_medium:
                severity = "Medium"
            else:
                severity = "High"

            if severity:
                cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 1)
                cv.putText(frame, f"%{round(score * 100, 2)} {label} ({severity} Severity)",
                           (box[0], box[1] - 10), cv.FONT_HERSHEY_COMPLEX, 0.5, (255, 0, 0), 1)

                pothole_data.append(f"Location: {g.latlng}, Bounding Box: {box}, Severity: {severity}")

    # Save output image
    output_image_path = os.path.join(result_path, 'detected_pothole.jpg')
    cv.imwrite(output_image_path, frame)

    # Write detected pothole details to the file
    with open(os.path.join(result_path, 'detected_pothole.txt'), 'w') as f:
        for data in pothole_data:
            f.write(f"{data}\n")
        f.write(f"\nTotal Pothole Area: {total_pothole_area} pixels\n")

    cv.imshow('Pothole Detection', frame)
    cv.waitKey(0)
    cv.destroyAllWindows()

except Exception as e:
    print(f"Error: {e}")


2026-07-11 00-39-23
Error: Failed to load image


## MAIN

In [3]:
import os
print(os.getcwd())

d:\PothHoleDetection_7


In [4]:
import cv2 as cv
import numpy as np
import time
import geocoder
import os
from datetime import datetime

try:
    x = datetime.now().strftime('%Y-%m-%d %H-%M-%S')
    print(x)

    class_name = []
    with open(r'utils/obj.names', 'r') as f:
        class_name = [cname.strip() for cname in f.readlines()]

    net1 = cv.dnn.readNet(r'utils/yolov4_tiny.weights', r'utils/yolov4_tiny.cfg')
    net1.setPreferableBackend(cv.dnn.DNN_BACKEND_CUDA)
    net1.setPreferableTarget(cv.dnn.DNN_TARGET_CUDA_FP16)
    model1 = cv.dnn_DetectionModel(net1)
    model1.setInputParams(size=(640, 480), scale=1/255, swapRB=True)
    cap = cv.VideoCapture(r"test.mp4")

    # Read the first frame to get the correct dimensions
    ret, frame = cap.read()
    if not ret:
        raise Exception("Failed to capture video")

    width = cap.get(3)
    height = cap.get(4)

    result = cv.VideoWriter('result.avi', cv.VideoWriter_fourcc(*'MJPG'), 10, (int(width), int(height)))
    g = geocoder.ip('me')
    result_path = "pothole_coordinates"
    starting_time = time.time()
    Conf_threshold = 0.5
    NMS_threshold = 0.4
    frame_counter = 0
    i = 0
    b = 0

    # Define the region of interest (ROI) mask
    mask = np.zeros_like(frame)
    mask[0:int(0.85*height), :] = 255


    while True:
        try:
            ret, frame = cap.read()
            frame_counter += 1
            if not ret:
                break

            # Apply the mask to the frame
            masked_frame = cv.bitwise_and(frame, mask)

            classes, scores, boxes = model1.detect(masked_frame, Conf_threshold, NMS_threshold)
            for (classid, score, box) in zip(classes, scores, boxes):
                label = "pothole"
                x, y, w, h = box
                recarea = w * h
                area = width * height

                severity = ""
                severity_threshold_low = 0.007  # Adjust as needed
                severity_threshold_medium = 0.020  # Adjust as needed

                if len(scores) != 0 and scores[0] >= 0.7:
                    if (recarea / area) <= severity_threshold_low:
                        severity = "Low"
                    elif (recarea / area) <= severity_threshold_medium:
                        severity = "Medium"
                    else:
                        severity = "High"

                    if severity != "":
                        cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 1)
                        cv.putText(frame, f"%{round(scores[0] * 100, 2)} {label} ({severity} Severity)",
                                   (box[0], box[1] - 10), cv.FONT_HERSHEY_COMPLEX, 0.5, (255, 0, 0), 1)

                        if i == 0:
                            cv.imwrite(os.path.join(result_path, 'pot' + str(i) + '.jpg'), frame)
                            with open(os.path.join(result_path, 'pot' + str(i) + '.txt'), 'w') as f:
                                f.write(f"{str(g.latlng)}\nSeverity: {severity}")
                                i = i + 1

                        if i != 0:
                            if (time.time() - b) >= 2:
                                cv.imwrite(os.path.join(result_path, 'pot' + str(i) + '.jpg'), frame)
                                with open(os.path.join(result_path, 'pot' + str(i) + '.txt'), 'w') as f:
                                    f.write(f"{str(g.latlng)}\nSeverity: {severity}")
                                    b = time.time()
                                    i = i + 1

            endingTime = time.time() - starting_time
            fps = frame_counter / endingTime
            cv.putText(frame, f'FPS: {fps}', (20, 50), cv.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)

            cv.imshow('frame', frame)
            result.write(frame)
            key = cv.waitKey(1)
            if key == ord('q'):
                break

        except Exception as e:
            print(f"Error: {e}")

except Exception as e:
    print(f"Error: {e}")

finally:
    cap.release()
    result.release()
    cv.destroyAllWindows()


2026-07-11 00-39-23
Error: Failed to capture video


NameError: name 'result' is not defined

# Streamlit

In [ ]:
import streamlit as st
import cv2 as cv
import numpy as np
import os
from datetime import datetime
import geocoder
from PIL import Image

# Streamlit UI
st.set_page_config(page_title="Pothole Detection App", layout="centered")
st.title("🛣️ Pothole Detection System")

# File uploader for user to upload an image
uploaded_file = st.file_uploader("Upload an image of the road:", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    # Read the uploaded image
    image = Image.open(uploaded_file)
    frame = np.array(image)

    # Ensure output directory exists
    result_path = "pothole_coordinates"
    os.makedirs(result_path, exist_ok=True)

    # Load YOLO model
    net = cv.dnn.readNet(r'utils/yolov4_tiny.weights', r'utils/yolov4_tiny.cfg')
    net.setPreferableBackend(cv.dnn.DNN_BACKEND_CUDA)
    net.setPreferableTarget(cv.dnn.DNN_TARGET_CUDA_FP16)
    model = cv.dnn_DetectionModel(net)
    model.setInputParams(size=(640, 480), scale=1/255, swapRB=True)

    height, width, _ = frame.shape
    Conf_threshold = 0.5
    NMS_threshold = 0.4

    # Define ROI (Region of Interest) Mask
    mask = np.zeros_like(frame)
    mask[0:int(0.85 * height), :] = 255
    masked_frame = cv.bitwise_and(frame, mask)

    # Detect potholes
    classes, scores, boxes = model.detect(masked_frame, Conf_threshold, NMS_threshold)
    g = geocoder.ip('me')  # Get location
    pothole_data = []
    total_pothole_area = 0

    for (classid, score, box) in zip(classes, scores, boxes):
        x, y, w, h = box
        recarea = w * h
        total_pothole_area += recarea
        image_area = width * height

        severity = ""
        if score >= 0.7:
            relative_area = recarea / image_area
            if relative_area <= 0.007:
                severity = "Low"
            elif relative_area <= 0.020:
                severity = "Medium"
            else:
                severity = "High"

            # Draw bounding box and label
            cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv.putText(frame, f"{round(score * 100, 2)}% Pothole ({severity})",
                       (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

            pothole_data.append(f"Location: {g.latlng}, Bounding Box: {box}, Severity: {severity}")

    # Save output image
    output_image_path = os.path.join(result_path, 'detected_pothole.jpg')
    cv.imwrite(output_image_path, frame)

    # Write details to a text file
    result_txt_path = os.path.join(result_path, 'detected_pothole.txt')
    with open(result_txt_path, 'w') as f:
        for data in pothole_data:
            f.write(f"{data}\n")
        f.write(f"\nTotal Pothole Area: {total_pothole_area} pixels\n")

    # Show output
    st.image(frame, caption="Detected Potholes", use_column_width=True)
    st.write(f"### 📏 Total Pothole Area: {total_pothole_area} pixels")

    # Provide download buttons
    with open(output_image_path, "rb") as file:
        btn = st.download_button(label="📥 Download Processed Image", data=file, file_name="detected_pothole.jpg", mime="image/jpeg")

    with open(result_txt_path, "rb") as file:
        btn = st.download_button(label="📄 Download Detection Report", data=file, file_name="detected_pothole.txt", mime="text/plain")

    st.success("Detection Complete! ✅")
